Author: Will Blanton


# Connections group-color classification


This experiment asks a narrower question than ranking: given a known valid four-word group, can the current semantic and puzzle-conditioned WordNet features classify it as purple, blue, green, or yellow?

The color labels come from the confirmed row order in `notes/connections-categories.md`: `purple, blue, green, yellow`. This dataset is balanced by construction, with one group per color per puzzle date. The original 16 words are available only to select puzzle-conditioned WordNet senses; the classifier receives one four-word group and no color-specific metadata.

Optuna selects hyperparameters by mean macro F1 across five date-level folds from the training split. A fresh final model then trains for up to 100 epochs on all primary training dates, using the retained validation split for early stopping. The notebook reports global accuracy, macro metrics, a confusion matrix, and precision/recall/F1/support for every color.


# Imports


In [1]:
import ast
import itertools
import random
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset, Subset

sys.path.insert(0, str(Path("src").resolve()))
from nyt_connections_solver.models import SemanticWordNetMixer, SetDecoder, SetEncoder
from nyt_connections_solver.wordnet import WordNetFeatureEncoder


# Constants and reproducibility


In [2]:
DATASET_PATH = Path("data/connections.csv")
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
COLORS_BY_ROW = ("purple", "blue", "green", "yellow")
COLOR_TO_INDEX = {color: index for index, color in enumerate(COLORS_BY_ROW)}
SEED = 0
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Data and features


In [3]:
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
# Correct known source formatting issues before parsing list literals.
connections_df.loc[1298, "connections"] = "['line', 'plane', 'point', 'solid']"
connections_df.loc[1892, "connections"] = "['abyss', 'fly', 'matrix', 'thing']"
connections_df = connections_df[~connections_df["date"].str.contains("04-01")].reset_index(drop=True)
connections_df["connections"] = connections_df["connections"].apply(ast.literal_eval)

dates = connections_df["date"].unique()
train_indices = random.sample(range(len(dates)), k=int(0.8 * len(dates)))
train_dates = dates[train_indices]
heldout_dates = np.random.permutation(dates[~np.isin(range(len(dates)), train_indices)])
split_index = len(heldout_dates) // 2
test_dates = heldout_dates[:split_index]
val_dates = heldout_dates[split_index:]
print(f"Dates: train={len(train_dates)}, validation={len(val_dates)}, test={len(test_dates)}")


Dates: train=968, validation=121, test=121


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
word_embedder = SentenceTransformer(EMBEDDING_MODEL, device=str(device))
word_embedder.requires_grad_(False)
word_embedder.eval()
SEMANTIC_DIM = int(word_embedder.get_sentence_embedding_dimension())

# This is the same active feature composition as the main LTR notebook:
# frozen semantic embedding followed by a puzzle-conditioned WordNet embedding.
wordnet_encoder = WordNetFeatureEncoder(word_embedder, max_senses=3, puzzle_conditioned=True)


class GroupFeatureBuilder:
    def __init__(self, word_embedder, wordnet_encoder):
        self.word_embedder = word_embedder
        self.wordnet_encoder = wordnet_encoder
        self.semantic_dim = int(word_embedder.get_sentence_embedding_dimension())
        self.wordnet_dim = wordnet_encoder.dim
        self.dim = self.semantic_dim + self.wordnet_dim

    def encode_full_puzzle(self, words):
        if len(words) != 16 or len(set(words)) != 16:
            raise ValueError("Feature construction requires 16 distinct puzzle words")
        semantic = self.word_embedder.encode(list(words), convert_to_numpy=True, show_progress_bar=False)
        wordnet = self.wordnet_encoder.encode(words)
        features = np.concatenate([semantic, wordnet], axis=1).astype(np.float32)
        if features.shape != (16, self.dim):
            raise ValueError(f"Expected (16, {self.dim}) features, got {features.shape}")
        return torch.from_numpy(features)


feature_builder = GroupFeatureBuilder(word_embedder, wordnet_encoder)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/tmp/ipykernel_52933/2594601848.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  SEMANTIC_DIM = int(word_embedder.get_sentence_embedding_dimension())
/mnt/code/python_projects/nyt-connections-solver/src/nyt_connections_solver/wordnet.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(embedder.get_sentence_embedding_dimension())
/tmp/ipykernel_52933/2594601848.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.semantic_dim = int(word_embedder.get_sentence_embedding_dimension())


In [5]:
class GroupColorDataset(Dataset):
    """One labeled valid group per puzzle row, with features conditioned on its puzzle."""

    def __init__(self, puzzle_df, feature_builder):
        self.examples = []
        for date, groups in puzzle_df.groupby("date")["connections"].agg(list).items():
            if len(groups) != 4 or any(len(group) != 4 for group in groups):
                raise ValueError(f"Puzzle {date} must contain four four-word groups")
            full_words = list(itertools.chain.from_iterable(groups))
            if len(set(full_words)) != 16:
                raise ValueError(f"Puzzle {date} contains repeated words")
            full_features = feature_builder.encode_full_puzzle(full_words)  # (16, feature_dim)
            for row, (color, words) in enumerate(zip(COLORS_BY_ROW, groups)):
                start = row * 4
                self.examples.append({
                    "x": full_features[start:start + 4],  # (4, feature_dim)
                    "y": torch.tensor(COLOR_TO_INDEX[color], dtype=torch.long),
                    "date": date,
                    "color": color,
                    "word_list": tuple(words),
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_groups(batch):
    return {
        "x": torch.stack([item["x"] for item in batch]),  # (B, 4, feature_dim)
        "y": torch.stack([item["y"] for item in batch]),  # (B,)
        "date": [item["date"] for item in batch],
        "color": [item["color"] for item in batch],
        "word_list": [item["word_list"] for item in batch],
    }


def make_loader(dates, *, shuffle):
    dataset = GroupColorDataset(
        connections_df.loc[connections_df["date"].isin(dates)], feature_builder
    )
    return dataset, DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate_groups)


train_dataset, train_loader = make_loader(train_dates, shuffle=True)
val_dataset, val_loader = make_loader(val_dates, shuffle=False)
test_dataset, test_loader = make_loader(test_dates, shuffle=False)


def loader_for_dates(dataset, selected_dates, *, shuffle):
    selected_dates = set(selected_dates)
    indices = [index for index, example in enumerate(dataset.examples) if example["date"] in selected_dates]
    if not indices:
        raise ValueError("A cross-validation split has no groups")
    return DataLoader(
        Subset(dataset, indices), batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate_groups
    )


fold_splitter = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_folds = []
for fold_index, (fold_train_indices, fold_val_indices) in enumerate(fold_splitter.split(train_dates)):
    fold_train_dates = train_dates[fold_train_indices]
    fold_val_dates = train_dates[fold_val_indices]
    cv_folds.append((
        fold_index,
        loader_for_dates(train_dataset, fold_train_dates, shuffle=True),
        loader_for_dates(train_dataset, fold_val_dates, shuffle=False),
    ))

print(
    f"Groups: train={len(train_dataset)}, validation={len(val_dataset)}, test={len(test_dataset)}; "
    f"CV folds={len(cv_folds)}"
)


Groups: train=3872, validation=484, test=484; CV folds=5


# Group-color classifier


In [6]:
class GroupColorClassifier(nn.Module):
    """The main Set Transformer feature path with a four-way group classifier head."""

    def __init__(
        self,
        *,
        in_dim,
        semantic_dim,
        wordnet_dim,
        attn_width=512,
        attn_layers=1,
        attn_heads=8,
        classifier_width=512,
        dropout=0.1,
    ):
        super().__init__()
        if semantic_dim + wordnet_dim != in_dim:
            raise ValueError("semantic_dim + wordnet_dim must equal in_dim")
        self.input_mixer = SemanticWordNetMixer(semantic_dim, wordnet_dim, attn_width)
        self.encoder = SetEncoder(
            attn_width,
            [attn_width] * attn_layers,
            attn_layers,
            [attn_heads] * attn_layers,
            dropout=dropout,
        )
        self.decoder = SetDecoder(
            in_dim=attn_width,
            h1=attn_width,
            out_dim=attn_width,
            num_seeds=1,
            mab_attn_heads=attn_heads,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Linear(attn_width, classifier_width),
            nn.LayerNorm(classifier_width),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_width, len(COLORS_BY_ROW)),
        )

    def forward(self, x):
        # x: (B, 4, semantic_dim + wordnet_dim)
        x = self.input_mixer(x)  # (B, 4, attn_width)
        x = self.encoder(x)  # (B, 4, attn_width)
        x = self.decoder(x).squeeze(1)  # (B, attn_width)
        return self.classifier(x)  # (B, 4)


# Training and metrics


In [7]:
def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    precision, recall, f1, support = precision_recall_fscore_support(
        labels,
        predictions,
        labels=np.arange(len(COLORS_BY_ROW)),
        zero_division=0,
    )
    per_color = pd.DataFrame({
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    }, index=COLORS_BY_ROW)
    return {
        "accuracy": float((labels == predictions).mean()),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "per_color": per_color,
        "confusion": pd.DataFrame(
            confusion_matrix(labels, predictions, labels=np.arange(len(COLORS_BY_ROW))),
            index=pd.Index(COLORS_BY_ROW, name="actual"),
            columns=pd.Index(COLORS_BY_ROW, name="predicted"),
        ),
    }


def evaluate_classifier(model, loader, criterion, device):
    model.eval()
    total_loss, count = 0.0, 0
    labels, predictions = [], []
    with torch.inference_mode():
        for batch in loader:
            x, y = batch["x"].to(device), batch["y"].to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * len(y)
            count += len(y)
            labels.extend(y.cpu().tolist())
            predictions.extend(logits.argmax(dim=-1).cpu().tolist())
    metrics = classification_metrics(labels, predictions)
    metrics["loss"] = total_loss / count
    return metrics


def format_global_metrics(prefix, metrics):
    return (
        f"{prefix} loss: {metrics['loss']:.4f} | {prefix} accuracy: {metrics['accuracy']:.1%} | "
        f"{prefix} macro precision: {metrics['macro_precision']:.3f} | "
        f"{prefix} macro recall: {metrics['macro_recall']:.3f} | "
        f"{prefix} macro F1: {metrics['macro_f1']:.3f}"
    )


def train_classifier(model, train_loader, val_loader, *, epochs, lr, weight_decay, patience=15):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    best_metric, best_state, stale_epochs = -float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        total_loss = total_correct = total_count = 0
        for batch in train_loader:
            x, y = batch["x"].to(device), batch["y"].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            total_correct += (logits.argmax(dim=-1) == y).sum().item()
            total_count += len(y)

        val_metrics = evaluate_classifier(model, val_loader, criterion, device)
        scheduler.step(val_metrics["loss"])
        if val_metrics["macro_f1"] > best_metric:
            best_metric = val_metrics["macro_f1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch} | Train loss: {total_loss / total_count:.4f} | "
                f"Train accuracy: {total_correct / total_count:.1%} | "
                + format_global_metrics("Validation", val_metrics)
            )
        if stale_epochs >= patience:
            print(f"Early stopping after {epoch + 1} epochs")
            break

    model.load_state_dict(best_state)
    return model.to("cpu"), {"best_epoch": epoch - stale_epochs, "best_macro_f1": best_metric}


# Five-fold tuned Set Transformer classifier


In [8]:
OPTUNA_TRIALS = 12
OPTUNA_EPOCHS = 100
OPTUNA_EARLY_STOPPING_PATIENCE = 15
FINAL_EPOCHS = 100
FINAL_EARLY_STOPPING_PATIENCE = 15
TUNING_SEED = SEED
trial_model_configs = {}


def seed_run(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def objective(trial):
    config = {
        "in_dim": feature_builder.dim,
        "semantic_dim": feature_builder.semantic_dim,
        "wordnet_dim": feature_builder.wordnet_dim,
        "attn_width": trial.suggest_categorical("attn_width", [256, 384, 512]),
        "attn_layers": trial.suggest_int("attn_layers", 1, 3),
        "attn_heads": 8,
        "classifier_width": trial.suggest_categorical("classifier_width", [256, 512, 768]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.3),
    }
    lr = trial.suggest_float("lr", 1e-5, 2e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    fold_metrics = []
    for fold_index, fold_train_loader, fold_val_loader in cv_folds:
        seed_run(TUNING_SEED + fold_index)
        fold_model, summary = train_classifier(
            GroupColorClassifier(**config),
            fold_train_loader,
            fold_val_loader,
            epochs=OPTUNA_EPOCHS,
            lr=lr,
            weight_decay=weight_decay,
            patience=OPTUNA_EARLY_STOPPING_PATIENCE,
        )
        metrics = evaluate_classifier(
            fold_model, fold_val_loader, nn.CrossEntropyLoss(), torch.device("cpu")
        )
        metrics["best_epoch"] = summary["best_epoch"]
        fold_metrics.append(metrics)
        if device.type == "cuda":
            torch.cuda.empty_cache()

    mean_macro_f1 = float(np.mean([metrics["macro_f1"] for metrics in fold_metrics]))
    mean_accuracy = float(np.mean([metrics["accuracy"] for metrics in fold_metrics]))
    trial.set_user_attr("fold_macro_f1", [metrics["macro_f1"] for metrics in fold_metrics])
    trial.set_user_attr("fold_accuracy", [metrics["accuracy"] for metrics in fold_metrics])
    trial.set_user_attr("fold_best_epochs", [metrics["best_epoch"] for metrics in fold_metrics])
    trial.set_user_attr("mean_cv_macro_f1", mean_macro_f1)
    trial.set_user_attr("mean_cv_accuracy", mean_accuracy)
    trial_model_configs[trial.number] = config
    return mean_macro_f1


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
    # Every trial completes every fold so scores are comparable.
    pruner=optuna.pruners.NopPruner(),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
best_model_config = trial_model_configs[best_trial.number]

# This is the final non-fold model: it uses all primary training dates, with the
# retained validation dates only for early stopping, then touches test once.
seed_run(TUNING_SEED)
model, final_training_summary = train_classifier(
    GroupColorClassifier(**best_model_config),
    train_loader,
    val_loader,
    epochs=FINAL_EPOCHS,
    lr=best_trial.params["lr"],
    weight_decay=best_trial.params["weight_decay"],
    patience=FINAL_EARLY_STOPPING_PATIENCE,
)
criterion = nn.CrossEntropyLoss()
val_metrics = evaluate_classifier(model, val_loader, criterion, torch.device("cpu"))
test_metrics = evaluate_classifier(model, test_loader, criterion, torch.device("cpu"))

print("Best mean CV macro F1:", best_trial.user_attrs["mean_cv_macro_f1"])
print("Best mean CV accuracy:", best_trial.user_attrs["mean_cv_accuracy"])
print("Final training best epoch:", final_training_summary["best_epoch"])
print(format_global_metrics("Final validation", val_metrics))
display(val_metrics["per_color"])
display(val_metrics["confusion"])
print(format_global_metrics("Final test", test_metrics))
display(test_metrics["per_color"])
display(test_metrics["confusion"])
optuna_results = study.trials_dataframe()
optuna_results


[I 2026-10-11 17:55:57,116] A new study created in memory with name: no-name-03fb5476-36ae-4d60-814e-ca180ac29f2f


  0%|          | 0/12 [00:00<?, ?it/s]

Epoch 0 | Train loss: 1.4068 | Train accuracy: 26.2% | Validation loss: 1.3637 | Validation accuracy: 31.7% | Validation macro precision: 0.163 | Validation macro recall: 0.317 | Validation macro F1: 0.212
Epoch 5 | Train loss: 1.2567 | Train accuracy: 39.9% | Validation loss: 1.3480 | Validation accuracy: 31.3% | Validation macro precision: 0.299 | Validation macro recall: 0.313 | Validation macro F1: 0.252
Epoch 10 | Train loss: 0.9545 | Train accuracy: 56.6% | Validation loss: 1.6228 | Validation accuracy: 31.4% | Validation macro precision: 0.311 | Validation macro recall: 0.314 | Validation macro F1: 0.310
Epoch 15 | Train loss: 0.7579 | Train accuracy: 65.8% | Validation loss: 1.8477 | Validation accuracy: 31.7% | Validation macro precision: 0.330 | Validation macro recall: 0.317 | Validation macro F1: 0.319
Epoch 20 | Train loss: 0.7302 | Train accuracy: 67.8% | Validation loss: 1.9168 | Validation accuracy: 31.6% | Validation macro precision: 0.328 | Validation macro recall: 0.

,precision,recall,f1,support
purple,0.357895,0.280992,0.314815,121
blue,0.284810,0.371901,0.322581,121
green,0.267717,0.280992,0.274194,121
yellow,0.423077,0.363636,0.391111,121


predicted,purple,blue,green,yellow
actual,,,,
purple,34,38,30,19
blue,29,45,31,16
green,24,38,34,25
yellow,8,37,32,44


Final test loss: 1.7939 | Final test accuracy: 36.2% | Final test macro precision: 0.374 | Final test macro recall: 0.362 | Final test macro F1: 0.363


,precision,recall,f1,support
purple,0.467391,0.355372,0.403756,121
blue,0.298851,0.429752,0.352542,121
green,0.280374,0.247934,0.263158,121
yellow,0.450450,0.413223,0.431034,121


predicted,purple,blue,green,yellow
actual,,,,
purple,43,55,15,8
blue,22,52,28,19
green,16,41,30,34
yellow,11,26,34,50


,number,value,datetime_start,datetime_complete,duration,params_attn_layers,params_attn_width,params_classifier_width,params_dropout,params_lr,params_weight_decay,user_attrs_fold_accuracy,user_attrs_fold_best_epochs,user_attrs_fold_macro_f1,user_attrs_mean_cv_accuracy,user_attrs_mean_cv_macro_f1,system_attrs_tpe:relative_params:0,state
0,0,0.330362,2026-10-11 17:55:57.123177,2026-10-11 17:56:54.956545,0 days 00:00:57.833368,2,384,512,0.272943,0.000179,0.000014,"[0.3286082474226804, 0.31829896907216493, 0.35...","[13, 22, 16, 10, 17]","[0.3307820566985442, 0.31729097641424386, 0.34...",0.331609,0.330362,NaN,COMPLETE
1,1,0.332644,2026-10-11 17:56:54.957843,2026-10-11 17:57:57.823544,0 days 00:01:02.865701,3,256,512,0.258155,0.000103,0.000407,"[0.3427835051546392, 0.31958762886597936, 0.36...","[9, 10, 8, 20, 19]","[0.34329498364339484, 0.3160357091435549, 0.35...",0.335980,0.332644,NaN,COMPLETE
2,2,0.332434,2026-10-11 17:57:57.824755,2026-10-11 17:58:57.958263,0 days 00:01:00.133508,3,256,512,0.286167,0.000048,0.000018,"[0.3389175257731959, 0.33505154639175255, 0.37...","[20, 9, 5, 15, 11]","[0.3407296085174073, 0.3289619455255785, 0.346...",0.340360,0.332434,NaN,COMPLETE
3,3,0.331699,2026-10-11 17:58:57.959390,2026-10-11 17:59:55.355270,0 days 00:00:57.395880,2,384,512,0.204233,0.000169,0.000111,"[0.3286082474226804, 0.3170103092783505, 0.346...","[25, 10, 16, 18, 10]","[0.3309590839463191, 0.3123824934504546, 0.345...",0.333169,0.331699,NaN,COMPLETE
4,4,0.319418,2026-10-11 17:59:55.356276,2026-10-11 18:00:31.247348,0 days 00:00:35.891072,1,512,512,0.082232,0.000026,0.000012,"[0.32216494845360827, 0.30670103092783507, 0.3...","[10, 11, 3, 7, 11]","[0.3231782880221874, 0.3072325884674055, 0.327...",0.323602,0.319418,NaN,COMPLETE
5,5,0.333313,2026-10-11 18:00:31.248330,2026-10-11 18:01:11.228437,0 days 00:00:39.980107,1,512,768,0.113323,0.000040,0.000005,"[0.3247422680412371, 0.3440721649484536, 0.364...","[20, 6, 5, 12, 9]","[0.3250164619519359, 0.31635105830173754, 0.35...",0.340639,0.333313,NaN,COMPLETE
6,6,0.340658,2026-10-11 18:01:11.229691,2026-10-11 18:01:46.208285,0 days 00:00:34.978594,1,512,768,0.074275,0.000123,0.000002,"[0.3337628865979381, 0.34536082474226804, 0.34...","[5, 14, 21, 1, 4]","[0.3287879390055133, 0.34760493878140936, 0.33...",0.346612,0.340658,NaN,COMPLETE
7,7,0.326983,2026-10-11 18:01:46.209307,2026-10-11 18:02:31.510579,0 days 00:00:45.301272,2,512,256,0.080049,0.000024,0.000002,"[0.327319587628866, 0.31572164948453607, 0.367...","[20, 10, 6, 7, 10]","[0.3264336450708692, 0.3119056459320151, 0.350...",0.333150,0.326983,NaN,COMPLETE
8,8,0.326500,2026-10-11 18:02:31.511575,2026-10-11 18:03:17.788981,0 days 00:00:46.277406,3,384,256,0.073485,0.000056,0.000614,"[0.3479381443298969, 0.3324742268041237, 0.329...","[11, 3, 7, 5, 8]","[0.34857004541322134, 0.3077247266058276, 0.33...",0.336259,0.326500,NaN,COMPLETE
9,9,0.336678,2026-10-11 18:03:17.789930,2026-10-11 18:04:04.082300,0 days 00:00:46.292370,3,384,768,0.055027,0.000120,0.000001,"[0.3402061855670103, 0.3131443298969072, 0.387...","[6, 12, 4, 6, 8]","[0.328969891950236, 0.3142055467607851, 0.3839...",0.341926,0.336678,NaN,COMPLETE


# Save classifier weights


In [9]:
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
weights_path = weights_dir / f"group_color_classifier_optuna_{timestamp}.pt"
checkpoint = {
    "model_class": "GroupColorClassifier",
    "model_config": best_model_config,
    "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "embedding_model_name": EMBEDDING_MODEL,
    "wordnet_max_senses": wordnet_encoder.max_senses,
    "wordnet_puzzle_conditioned": wordnet_encoder.puzzle_conditioned,
    "colors_by_row": list(COLORS_BY_ROW),
    "selection_metric": "mean_cv_macro_f1",
    "cross_validation": {"n_splits": 5, "epochs": OPTUNA_EPOCHS, "patience": OPTUNA_EARLY_STOPPING_PATIENCE},
    "final_training": {"epochs": FINAL_EPOCHS, "patience": FINAL_EARLY_STOPPING_PATIENCE, **final_training_summary},
    "split_dates": {
        name: pd.to_datetime(values).strftime("%Y-%m-%d").tolist()
        for name, values in (("train", train_dates), ("val", val_dates), ("test", test_dates))
    },
    "dataset_path": str(DATASET_PATH.resolve()),
}
torch.save(checkpoint, weights_path)
print(f"Saved group-color classifier checkpoint to {weights_path}")


Saved group-color classifier checkpoint to weights/group_color_classifier_optuna_20261011_180532.pt
